# Inferring gene expression from patch PCM images with PENNE

In this tutorial, you will learn how to infer gene expressions from PCM images. Please ensure that you have the following files before running the inference:
- A folder of images for inference: All the PCM patches that you want to use for inference. The examples we used here are the first 10 images from [Debier et al., 2005](https://doi.org/10.1109/TMI.2005.846851). You are more than welcome to use your own datasets.
- PENNE checkpoint: Checkpoint file for the PENNE model, in the ```.ckpt``` format saved by ```TrainPENNE```. See more in the [documentation](../documentations.md).
- SPAGHETTI checkpoint: Checkpoint file for SPAGHETTI model in ```.ckpt``` format. This is used for domain adaptation. For more details about using or training the SPAGHETTI model, see the official [SPAGHETTI repo](https://github.com/schwartzlab-methods/spaghetti).
- List of genes for inference (in ```.txt```): A list of genes that are used as the ```var``` list for the output ```AnnData``` object.
- High confidence gene (optional, in ```.txt```): A list of genes that are considered high confidence. This list is used to subset the ```AnnData``` object.

You can generate all those files by training your own models for SPAGHETTI and PENNE, or you can use the existing ones for the base version of the model located at ```penne/assets/```. 

## Installing PENNE

Make sure you install ```penne``` using the steps listed in [README.md](../README.md). Once that's done, run the following code to see if it has installed correctly:

In [1]:
import penne

print(penne.__version__)

/Users/ric/opt/anaconda3/envs/penne/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.0.0


## Preparing the datasets

We first need to prepare the inference dataset to be fed into PENNE. To do this, we use the ```InferenceDataset``` class. This class takes either a single directory ```str``` to the images, or multiple directories in ```tuple[str]``` or ```list[str]``` of the images. We also want to wrap this dataset in a ```DataLoader``` class.

In [2]:
from penne.dataset import InferenceDataset

dataset = InferenceDataset(
    paths="inference_images/pcm"
)
print(dataset.images)


['inference_images/pcm/exp0009.jpg', 'inference_images/pcm/exp0008.jpg', 'inference_images/pcm/exp0001.jpg', 'inference_images/pcm/exp0003.jpg', 'inference_images/pcm/exp0002.jpg', 'inference_images/pcm/exp0006.jpg', 'inference_images/pcm/exp0007.jpg', 'inference_images/pcm/exp0005.jpg', 'inference_images/pcm/exp0004.jpg', 'inference_images/pcm/exp0010.jpg']


In [3]:
# put in DataLoader
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
print(len(dataloader)) # the number of batches, should be the same as the number of images times batch size

10


## Running Inference

We now start running the inference by passing this dataset to PENNE. We need to first initialize the ```Penne``` class, then just pass the datasets into PENNE. PENNE will automatically determine if a GPU is available and use it when possible, so you all have to do is to just run it!

In [4]:
import penne
from penne.model import Penne

# intialize the model
# you can specify the path to the model weights if you have them locally. 
# Otherwise it will use the ones stored in assets/ as default (the version that is pubished in the paper is stored in assets/spaghetti.ckpt)
# this will automatically download the weights from GitHub if they are not found in the local path at assets/.
model = Penne() 
print(model)

Penne(
  (model): TrainPenne(
    (feature_translator): OrthogonalTranslator(
      (fc1): Linear(in_features=1024, out_features=512, bias=True)
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (fc2): Linear(in_features=512, out_features=512, bias=True)
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (fc3): Linear(in_features=512, out_features=1024, bias=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (domain_separator): DomainDiscriminator(
      (model): Sequential(
        (0): Linear(in_features=64, out_features=512, bias=True)
        (1): ReLU()
        (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (3): Dropout(p=0.3, inplace=False)
        (4): Linear(in_features=512, out_features=256, bias=True)
        (5): ReLU()
        (6): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (7): Dropout(p=0.3, inplace=False)
        (8): Linear(in_features=256, out_features=128, bias=True)
      

In [5]:
res = model(dataloader)

print(res)

100%|██████████| 10/10 [00:34<00:00,  3.45s/it]

AnnData object with n_obs × n_vars = 10 × 18085
    obs: 'image_name'
    var: 'gene_name'


In [6]:
print("========obs=========")
print(res.obs)
print("========var=========")
print(res.var)

========obs=========
    image_name
0  exp0009.jpg
1  exp0008.jpg
2  exp0001.jpg
3  exp0003.jpg
4  exp0002.jpg
5  exp0006.jpg
6  exp0007.jpg
7  exp0005.jpg
8  exp0004.jpg
9  exp0010.jpg
========var=========
      gene_name
0          A1CF
1           A2M
2         A2ML1
3       A3GALT2
4        A4GALT
...         ...
18080      ZXDC
18081    ZYG11A
18082    ZYG11B
18083       ZYX
18084     ZZEF1

[18085 rows x 1 columns]
